# Task B — Novel-View / Frame Synthesis for a Camera Rig

**Yandex ML Cup — ML track**

**Goal:** reconstruct what one camera on a self-driving car saw at an intermediate moment, given the surrounding frames in time plus a dense LiDAR point cloud.

**Metric:** PSNR of the predicted image vs. ground truth (higher = better) → mapped to a competition score.

---

## The setup

The car has a **6-camera rig**: `front`, `left_fwd`, `left_bwd`, `right_fwd`, `right_bwd`, `rear`.

For each sample we get, from `meta.json`:
- Frames from all cameras at time **t0** and time **t1** (2 s apart)
- A **target camera** and a **target timestamp** *between* t0 and t1
- Camera intrinsics + extrinsics (camera-to-world poses)
- A **dense LiDAR** aggregation (~10M points, 55 sweeps) in world coordinates

**We must predict the target camera's image at the target time** — a frame that was never recorded.

## The strategy — blend three independent estimates

No single method is reliable everywhere, so the solution produces **three estimates** of the target frame and blends them with confidence weights:

| Estimate | Signal used | Strong where |
|---|---|---|
| **1. Temporal optical-flow interpolation** | the same camera at t0 & t1 | smooth motion, static-ish scene |
| **2. RIFE deep frame interpolation** | t0 & t1 frames (neural) | non-linear motion, deformation |
| **3. LiDAR image-based rendering** | 3D geometry + other cameras | geometry / parallax, occlusions |

Final blend weights (tuned): `temporal_strength=0.72`, `rife_mix=0.42`, `lidar_mix=0.30`.

## Estimate 1 — temporal optical-flow interpolation

The target is a fraction `α` of the way from t0 to t1. If we know how each pixel *moves* from frame 0 to frame 1, we can slide it `α` of the way there.

1. Compute dense optical flow **both ways** with OpenCV **DIS** flow: `f01` (0→1) and `f10` (1→0).
2. Warp frame 0 forward by `α·f01`, warp frame 1 backward by `(1−α)·f10`.
3. Blend the two warps. Where they **disagree** (occlusion), trust the side whose warp error is lower — a soft occlusion mask.

Forward-and-backward flow with a disagreement mask is what handles objects that appear/disappear between frames.

In [ ]:
import cv2
import numpy as np

def flow_dis(a, b):
    """Dense optical flow a->b using OpenCV DIS (fast, robust)."""
    ga = cv2.cvtColor(a, cv2.COLOR_RGB2GRAY)
    gb = cv2.cvtColor(b, cv2.COLOR_RGB2GRAY)
    dis = cv2.DISOpticalFlow_create(cv2.DISOPTICAL_FLOW_PRESET_MEDIUM)
    return dis.calc(ga, gb, None)   # HxWx2 (dx, dy) per pixel

def remap_image(img, flow_xy, scale):
    """Warp img along a scaled flow field."""
    h, w = img.shape[:2]
    yy, xx = np.mgrid[0:h, 0:w].astype(np.float32)
    map_x = xx + scale * flow_xy[..., 0]
    map_y = yy + scale * flow_xy[..., 1]
    return cv2.remap(img, map_x, map_y, cv2.INTER_LINEAR, borderMode=cv2.BORDER_REFLECT)

def temporal_interpolate(img0, img1, alpha):
    f01 = flow_dis(img0, img1)
    f10 = flow_dis(img1, img0)
    warp0 = remap_image(img0, f01, alpha)          # push frame 0 forward
    warp1 = remap_image(img1, f10, 1.0 - alpha)    # push frame 1 backward
    # occlusion-aware blend: weight toward the lower-error side
    err0 = np.abs(warp0.astype(np.float32) - img1).mean(-1, keepdims=True)
    err1 = np.abs(warp1.astype(np.float32) - img0).mean(-1, keepdims=True)
    w0 = (1 - alpha) * (err1 + 1e-3)
    w1 = alpha * (err0 + 1e-3)
    return ((w0 * warp0 + w1 * warp1) / (w0 + w1)).astype(np.uint8)

# temporal_interpolate(frame_t0, frame_t1, alpha=0.5)  -> predicted middle frame

## Estimate 2 — RIFE deep frame interpolation

[RIFE](https://github.com/megvii-research/ECCV2022-RIFE) is a neural network built exactly for "given frame 0 and frame 1, synthesize the frame at time `α`." It learns motion + occlusion end-to-end and handles non-linear, deforming motion that hand-tuned optical flow smears.

It's **mixed in** rather than used alone (`rife_mix=0.42`) — RIFE is great on motion but can hallucinate texture, so averaging it with the flow estimate keeps things stable.

## Estimate 3 — LiDAR image-based rendering (the geometry path)

The two estimates above are pure 2D. But we also have **true 3D geometry** from LiDAR — this is what fixes parallax (near objects shift more than far ones as the car moves).

The classic projective-geometry pipeline:

1. **Project** the LiDAR cloud into the target camera → a dense **depth map** of the target view.
2. For each target pixel, **back-project** it to a 3D world point using that depth.
3. **Re-project** that world point into a *source* camera (t0/t1, possibly a different camera on the rig) and **sample its color** (bilinear).
4. Weight by confidence (how head-on the surface is, how close the depth match is).

Projection uses the pinhole model with distortion: $u = K \, [R \mid t]^{-1} X_{world}$.

In [ ]:
def world_to_camera(xyz, c2w):
    """World points -> camera frame using camera-to-world pose (invert it)."""
    R, t = c2w[:3, :3], c2w[:3, 3]
    return (xyz - t) @ R          # R is orthonormal, so R^{-1} == R^T

def project(xyz_world, c2w, K):
    """Project world points into pixel coords of a camera. Returns u, v, depth."""
    cam = world_to_camera(xyz_world, c2w)
    z = cam[:, 2]
    valid = z > 1e-3                          # keep points in front of the camera
    uv = (K @ cam.T).T
    u = uv[:, 0] / uv[:, 2]
    v = uv[:, 1] / uv[:, 2]
    return u, v, z, valid

def build_depth_map(lidar_xyz, target_c2w, target_K, h, w):
    """Splat lidar into the target view, keeping the nearest point per pixel (z-buffer)."""
    u, v, z, valid = project(lidar_xyz, target_c2w, target_K)
    depth = np.full((h, w), np.inf, np.float32)
    ui, vi = np.round(u).astype(int), np.round(v).astype(int)
    m = valid & (ui >= 0) & (ui < w) & (vi >= 0) & (vi < h)
    for uu, vv, zz in zip(ui[m], vi[m], z[m]):   # nearest wins (z-buffer)
        if zz < depth[vv, uu]:
            depth[vv, uu] = zz
    depth[np.isinf(depth)] = 0.0
    return depth

# In the real solution this is vectorized + densified (holes between sparse
# lidar returns are filled) before sampling colors from the source cameras.

## Putting it together

```
               ┌─────────────────────────────┐
  t0, t1  ───▶ │ 1. DIS optical-flow interp  │──┐
  frames       └─────────────────────────────┘  │
               ┌─────────────────────────────┐  │   weighted
  t0, t1  ───▶ │ 2. RIFE neural interp        │──┼──▶ blend ──▶ target frame
  frames       └─────────────────────────────┘  │   (by conf.)
               ┌─────────────────────────────┐  │
  lidar +  ──▶ │ 3. LiDAR image-based render  │──┘
  poses        └─────────────────────────────┘
```

Everything is computed at the target resolution, blended per-pixel by confidence, then written as high-quality JPEG (`quality=95, subsampling=0`) since the metric is pixel-accurate PSNR.

## Why blend instead of pick one?

PSNR punishes any region that's badly wrong. Each method fails differently:

- **Optical flow** smears fast / non-linear motion and thin structures.
- **RIFE** can hallucinate plausible-but-wrong texture.
- **LiDAR IBR** is geometrically correct but has holes (sparse returns, occlusion) and mis-colors at grazing angles.

A confidence-weighted average lets each method cover the others' failure regions — the classic reason ensembles win on a squared-error-style metric.

## Summary

- **Task:** synthesize an unrecorded camera frame at an intermediate time, scored by PSNR.
- **Three complementary estimates:** DIS optical-flow interpolation, RIFE neural interpolation, and LiDAR-driven image-based rendering.
- **Geometry matters:** projecting a dense LiDAR cloud into the target view and re-sampling source cameras is what handles parallax and occlusion that pure 2D interpolation can't.
- **Confidence-weighted blend** of the three beats any single method because their failure modes don't overlap.
- **Metric-aware output:** near-lossless JPEG, because every pixel counts toward PSNR.